# AI Sentiment Intelligence
## AI-Based Sentiment Analysis System Using Machine Learning
**Academic Minor Project** | Final Deadline: 11 September 2026

### STAGE 2: DATASET + NLP PREPROCESSING ENGINE
This notebook implements an end-to-end data engineering and Natural Language Processing (NLP) preprocessing pipeline that transforms raw sentiment text into high-quality, normalized tokens suitable for TF-IDF feature extraction and machine learning classifiers in Stage 3.

**Target Polarities:**
* **Positive**
* **Negative**
* **Neutral**


## 1. Import Libraries
Load core scientific, visualization, and custom modular source packages.


In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path for modular imports
sys.path.insert(0, os.path.abspath('..'))

import config
from src.data_loader import (
    load_dataset,
    get_default_dataset_path,
    validate_dataset,
    format_validation_summary,
    report_dataset_info,
    calculate_text_statistics,
    plot_sentiment_distribution,
    plot_text_length_distribution,
    generate_data_quality_report,
)
from src.data_cleaning import (
    clean_text_basic,
    safe_str,
    to_lowercase,
    remove_urls,
    remove_html_tags,
    remove_emails,
    remove_special_characters,
    remove_extra_whitespace,
    clean_dataset,
)
from src.preprocessing import (
    expand_contractions,
    tokenize_text,
    get_controlled_stopwords,
    remove_stopwords,
    lemmatize_tokens,
    preprocess_text,
    preprocess_dataset,
    analyze_word_frequencies,
    plot_word_frequency_analysis,
)

# Set plotting aesthetics
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print("All libraries and modular project components imported successfully!")


All libraries and modular project components imported successfully!


## 2. Project Configuration
Centralized paths, supported classes, and NLP configuration parameters.


In [2]:
config.ensure_directories()

print(f"Base Directory:        {config.BASE_DIR}")
print(f"Raw Data Path:         {config.RAW_DATA_PATH}")
print(f"Processed Data Path:   {config.PROCESSED_DATA_PATH}")
print(f"Results Path:          {config.RESULTS_PATH}")
print(f"Supported Classes:     {config.SUPPORTED_LABELS}")
print(f"Random State:          {config.RANDOM_STATE}")
print(f"Preserved Negations:   {sorted(list(config.PRESERVED_WORDS))}")


Base Directory:        C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence
Raw Data Path:         C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\data\raw
Processed Data Path:   C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\data\processed
Results Path:          C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results
Supported Classes:     ['Positive', 'Negative', 'Neutral']
Random State:          42
Preserved Negations:   ['against', 'barely', 'hardly', 'neither', 'never', 'no', 'nor', 'not', 'nothing', 'nowhere', 'scarcely', 'without']


## 3. Dataset Loading
Locates and loads the sentiment dataset.
The system automatically resolves the dataset path, prioritizing real user data if available or the Stage 2 benchmark demo dataset.


In [3]:
dataset_path = get_default_dataset_path()
print(f"Resolved Dataset Path: {dataset_path.resolve()}")

df_raw = load_dataset(dataset_path)
print(f"Successfully loaded {len(df_raw)} records.")


Resolved Dataset Path: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\data\raw\demo_dataset.csv
[SUCCESS] Dataset loaded successfully from: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\data\raw\demo_dataset.csv (60 records)
Successfully loaded 60 records.


## 4. Dataset Dimensions and Structure
Inspect total rows, columns, and data types.


In [4]:
print(f"Dataset Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print("\nColumn Data Types:")
print(df_raw.dtypes)


Dataset Shape: 60 rows x 2 columns

Column Data Types:
text         object
sentiment    object
dtype: object


## 5. Dataset Preview
Examine the first 10 records from the raw dataset.


In [5]:
df_raw.head(10)


## 6. Comprehensive Data Quality Audit
Validates schema integrity, missing values, empty strings, duplicates, and label correctness.
Saves official audit reports to `results/data_quality_report.json` and `.txt`.


In [6]:
validation_results = validate_dataset(df_raw)
print(format_validation_summary(validation_results))

# Persist quality report
generate_data_quality_report(df_raw)


DATASET VALIDATION SUMMARY
Status: VALID
Dataset Shape: 60 rows, 2 columns
Columns: ['text', 'sentiment']

Missing Values:
  Text: 0
  Sentiment: 0

Duplicate Rows: 0
Empty / Whitespace-only Strings: 0
Invalid Sentiment Label Rows: 0

Sentiment Classes:
  Positive: 20
  Negative: 20
  Neutral: 20

Text Length Statistics:
  Char Length: min=59, avg=71.6, max=82
  Word Count:  min=8, avg=11.5, max=16

Quality Check: No data integrity issues detected.
[INFO] Data quality report saved to:
  - C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\data_quality_report.json
  - C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\data_quality_report.txt


## 7. Missing Value Analysis
Check and report missing values in each feature.


In [7]:
missing = df_raw.isnull().sum()
print("Missing values per column:")
print(missing)
print(f"\nTotal missing cells: {missing.sum()}")


Missing values per column:
text         0
sentiment    0
dtype: int64

Total missing cells: 0


## 8. Duplicate Record Analysis
Identify redundant entries to prevent train/test data leakage and classifier bias.


In [8]:
duplicate_count = df_raw.duplicated(subset=[config.TEXT_COLUMN]).sum()
print(f"Duplicate text records detected: {duplicate_count}")


Duplicate text records detected: 0


## 9. Sentiment Class Distribution
Analyze class balance across Positive, Negative, and Neutral categories.


In [9]:
class_counts = df_raw[config.SENTIMENT_COLUMN].value_counts()
print("Sentiment Class Counts:")
print(class_counts)

# Generate publication-quality visualization
dist_plot_path = plot_sentiment_distribution(df_raw)
print(f"Saved distribution plot to: {dist_plot_path}")


Sentiment Class Counts:
sentiment
Positive    20
Negative    20
Neutral     20
Name: count, dtype: int64
[INFO] Sentiment distribution plot saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\sentiment_distribution.png
Saved distribution plot to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\sentiment_distribution.png


## 10. Text Cleaning Pipeline
Demonstration of individual cleaning functions:
- Lowercase normalization
- URL removal
- HTML tag & entity stripping
- Email removal
- Whitespace trimming


In [10]:
sample_noisy_text = "Check OUT: https://test.com! Email: help@site.com. <b>AWESOME</b> item... loved it!   "
print("Raw Sample:     ", repr(sample_noisy_text))
print("Lowercased:     ", repr(to_lowercase(sample_noisy_text)))
print("No URLs:        ", repr(remove_urls(sample_noisy_text)))
print("No HTML:        ", repr(remove_html_tags(sample_noisy_text)))
print("No Emails:      ", repr(remove_emails(sample_noisy_text)))
print("Cleaned Basic:  ", repr(clean_text_basic(sample_noisy_text)))


Raw Sample:      'Check OUT: https://test.com! Email: help@site.com. <b>AWESOME</b> item... loved it!   '
Lowercased:      'check out: https://test.com! email: help@site.com. <b>awesome</b> item... loved it!   '
No URLs:         'Check OUT:  Email: help@site.com. <b>AWESOME</b> item... loved it!   '
No HTML:         'Check OUT: https://test.com! Email: help@site.com.  AWESOME  item... loved it!   '
No Emails:       'Check OUT: https://test.com! Email: . <b>AWESOME</b> item... loved it!   '
Cleaned Basic:   'check out: email: . awesome item... loved it!'


## 11. Contraction Handling & Negation Preservation
Contractions like *don't*, *wasn't*, and *couldn't* contain vital sentiment polarity.
Expanding them to *do not*, *was not*, and *could not* ensures the negation survives stopword filtering.


In [11]:
sample_contraction = "I didn't like the battery, won't recommend it, and it isn't worth the money."
print("Before Contraction Expansion:")
print(" ", sample_contraction)
print("\nAfter Contraction Expansion:")
print(" ", expand_contractions(sample_contraction))


Before Contraction Expansion:
  I didn't like the battery, won't recommend it, and it isn't worth the money.

After Contraction Expansion:
  I did not like the battery, will not recommend it, and it is not worth the money.


## 12. Word Tokenization
Splits normalized sentences into discrete word tokens using NLTK.


In [12]:
sample_sentence = "The customer support was friendly and resolved my issue in minutes."
tokens = tokenize_text(sample_sentence)
print("Input Sentence: ", sample_sentence)
print("Tokens:         ", tokens)
print(f"Token Count:    {len(tokens)}")


Input Sentence:  The customer support was friendly and resolved my issue in minutes.
Tokens:          ['the', 'customer', 'support', 'was', 'friendly', 'and', 'resolved', 'my', 'issue', 'in', 'minutes']
Token Count:    11


## 13. Controlled Stopword Strategy
Standard stopword lists remove *not* and *no*, which inverts sentiment meaning.
Our controlled stopword list explicitly preserves all negation terms.


In [13]:
controlled_stops = get_controlled_stopwords(exclude_negations=True)
print(f"Controlled stopword count: {len(controlled_stops)}")
print(f"Is 'not' in stopword filter?    {'not' in controlled_stops} (Preserved!)")
print(f"Is 'never' in stopword filter?  {'never' in controlled_stops} (Preserved!)")
print(f"Is 'the' in stopword filter?    {'the' in controlled_stops} (Filtered)")

sample_tokens = ["this", "laptop", "is", "not", "good", "and", "never", "works"]
filtered_tokens = remove_stopwords(sample_tokens)
print("\nOriginal Tokens: ", sample_tokens)
print("Filtered Tokens: ", filtered_tokens)


Controlled stopword count: 194
Is 'not' in stopword filter?    False (Preserved!)
Is 'never' in stopword filter?  False (Preserved!)
Is 'the' in stopword filter?    True (Filtered)

Original Tokens:  ['this', 'laptop', 'is', 'not', 'good', 'and', 'never', 'works']
Filtered Tokens:  ['laptop', 'not', 'good', 'never', 'works']


## 14. WordNet Lemmatization
Morphologically reduces inflectional variants (e.g. *loved*, *loving*, *loves* -> *love*) to their dictionary base form.


In [14]:
sample_words = ["loved", "loving", "loves", "crashes", "crashing", "batteries", "delivering"]
lemmatized = lemmatize_tokens(sample_words)

for original, lemma in zip(sample_words, lemmatized):
    print(f"  {original:12} -> {lemma}")


  loved        -> love
  loving       -> love
  loves        -> love
  crashes      -> crash
  crashing     -> crash
  batteries    -> battery
  delivering   -> deliver


## 15. Before vs. After Preprocessing (Actual Dataset Samples)
Examine real examples from the dataset transformed by the complete NLP pipeline.


In [15]:
print("Genuine Dataset Samples (Before vs. After Full Preprocessing):")
print("=" * 80)
for idx, row in df_raw.head(6).iterrows():
    original = row[config.TEXT_COLUMN]
    cleaned = preprocess_text(original)
    label = row[config.SENTIMENT_COLUMN]
    print(f"[{label.upper()}]")
    print(f"  Original: {original}")
    print(f"  Cleaned:  {cleaned}\n")
print("=" * 80)


Genuine Dataset Samples (Before vs. After Full Preprocessing):
[POSITIVE]
  Original: The product quality is exceptional and exceeded all my expectations.
  Cleaned:  product quality exceptional exceed expectation

[POSITIVE]
  Original: Customer support was quick, friendly, and resolved my issue in minutes.
  Cleaned:  customer support quick friendly resolve issue minute

[POSITIVE]
  Original: I really love the clean design and smooth user experience of this app.
  Cleaned:  really love clean design smooth user experience app

[POSITIVE]
  Original: This laptop has great battery life and delivers outstanding performance.
  Cleaned:  laptop great battery life deliver outstanding performance

[POSITIVE]
  Original: The delivery arrived ahead of schedule and the packaging was in perfect condition.
  Cleaned:  delivery arrive ahead schedule package perfect condition

[POSITIVE]
  Original: Highly recommend this service to anyone looking for reliable and fast solutions.
  Cleaned:  highly

## 16. Execute Full Dataset Preprocessing
Apply the complete pipeline across all dataset records.
Raw text and original labels are preserved, and preprocessed text is stored in `clean_text`.


In [16]:
# Remove nulls/duplicates from raw data for clean processing
df_clean = df_raw.dropna(subset=[config.TEXT_COLUMN, config.SENTIMENT_COLUMN]).drop_duplicates(subset=[config.TEXT_COLUMN]).copy()

df_processed = preprocess_dataset(
    df_clean,
    text_column=config.TEXT_COLUMN,
    new_column="clean_text",
    remove_empty_cleaned=True,
)

print(f"Preprocessed DataFrame shape: {df_processed.shape}")
df_processed.head(10)


[PREPROCESSING] Running NLP pipeline on 60 records in column 'text'...
[PREPROCESSING] Completed. Preprocessed column: 'clean_text'
Preprocessed DataFrame shape: (60, 3)


## 17. Text Length & Word Count Analysis
Analyze word counts and character counts to inform TF-IDF hyperparameter selection.


In [17]:
stats = calculate_text_statistics(df_processed, text_column=config.TEXT_COLUMN)
print("Overall Character Statistics:", stats['character_stats'])
print("Overall Word Count Statistics:", stats['word_stats'])
print("\nAverage Word Counts per Sentiment Class:")
for cls, cls_stat in stats['class_breakdown'].items():
    print(f"  {cls:10}: {cls_stat['avg_words']} words/sample (count={cls_stat['count']})")

# Generate visualization
length_plot_path = plot_text_length_distribution(df_processed)
print(f"Saved text length distribution plot to: {length_plot_path}")


Overall Character Statistics: {'min': 59, 'max': 82, 'mean': 71.6, 'median': 71.0}
Overall Word Count Statistics: {'min': 8, 'max': 16, 'mean': 11.5, 'median': 11.0}

Average Word Counts per Sentiment Class:
  Positive  : 11.65 words/sample (count=20)
  Negative  : 11.65 words/sample (count=20)
  Neutral   : 11.2 words/sample (count=20)
[INFO] Text length distribution plot saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\text_length_distribution.png
Saved text length distribution plot to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\text_length_distribution.png


## 18. Word Frequency Analysis per Sentiment Class
Extract and visualize top vocabulary tokens characteristic of each polarity.


In [18]:
freq_data = analyze_word_frequencies(
    df_processed,
    text_column="clean_text",
    sentiment_column=config.SENTIMENT_COLUMN,
    top_n=10,
    save_json_path=config.WORD_FREQ_JSON,
)

freq_plot_path = plot_word_frequency_analysis(freq_data, output_path=config.WORD_FREQ_PLOT)
print(f"Saved word frequency plot to: {freq_plot_path}")
for label, words in freq_data.items():
    print(f"  {label}: {list(words.keys())[:6]}")


[INFO] Word frequencies saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\word_frequencies.json
[INFO] Word frequency plot saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\word_frequency_analysis.png
Saved word frequency plot to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\word_frequency_analysis.png
  Positive: ['quality', 'customer', 'friendly', 'minute', 'user', 'great']
  Negative: ['not', 'completely', 'customer', 'service', 'take', 'day']
  Neutral: ['schedule', 'take', 'package', 'afternoon', 'standard', 'cm']


## 19. Save Preprocessed Dataset
Persists the final processed data to `data/processed/cleaned_dataset.csv`.
Columns: `text` (original), `sentiment` (original), `clean_text` (preprocessed).


In [19]:
output_df = df_processed[[config.TEXT_COLUMN, config.SENTIMENT_COLUMN, "clean_text"]]
output_df.to_csv(config.CLEANED_DATA_FILE, index=False)
print(f"[SUCCESS] Cleaned dataset saved to: {config.CLEANED_DATA_FILE.resolve()}")
print(f"Total records saved: {len(output_df)}")

# Verify raw dataset remains untouched
print(f"Raw dataset file exists: {dataset_path.exists()} (UNTOUCHED)")


[SUCCESS] Cleaned dataset saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\data\processed\cleaned_dataset.csv
Total records saved: 60
Raw dataset file exists: True (UNTOUCHED)


## 20. Stage 2 Summary & Transition to Stage 3
### Stage 2 Accomplishments:
1. **Dataset Ingestion & Validation**: Verified schema, zero missing values, zero duplicates.
2. **Text Cleaning**: Safely normalized casing, stripped URLs, HTML tags, and email noise.
3. **Contraction Handling**: Expanded 50+ common English contractions (e.g., *don't* -> *do not*).
4. **Tokenization & Controlled Stopwords**: Tokenized text while strictly preserving negation signals (*not*, *never*, *no*).
5. **Morphological Lemmatization**: Unified inflectional word forms (*loved* -> *love*).
6. **Artifact Generation**: Generated `cleaned_dataset.csv`, `data_quality_report.json`, and 3 publication-ready visualizations.

### Ready for Stage 3:
- **TF-IDF Feature Extraction**: Converting `clean_text` into numerical feature matrices.
- **Model Training**: Logistic Regression, Multinomial Naive Bayes, and Linear Support Vector Machines (SVM).


# Stage 3 - Feature Engineering & Model Training

In this stage, we transition from preprocessed text data to numerical feature representations using **TF-IDF Vectorization**, followed by training three distinct machine learning classifiers:
1. **Logistic Regression** (Linear probabilistic model)
2. **Multinomial Naive Bayes** (Probabilistic generative model based on Bayes' Theorem)
3. **Linear Support Vector Machine (Linear SVM)** (Max-margin hyperplane classifier)

### Zero Data Leakage Guarantee:
The dataset is split into training (80%) and testing (20%) sets using **stratified sampling** *before* fitting the TF-IDF vectorizer. The vectorizer is fitted exclusively on `X_train`. The test set `X_test` is only transformed using the learned training vocabulary.

## 1. Load Cleaned Dataset
We load the preprocessed dataset (`data/processed/cleaned_dataset.csv`) generated during Stage 2, containing `text`, `sentiment`, and `clean_text`.

In [45]:
# Load cleaned dataset produced in Stage 2
cleaned_file = config.CLEANED_DATA_FILE
df_clean = load_dataset(cleaned_file)
print(f"Dataset successfully loaded from: {cleaned_file}")
print(f"Total Rows: {df_clean.shape[0]}, Columns: {df_clean.shape[1]}")
display_cols = [c for c in ['text', 'sentiment', 'clean_text'] if c in df_clean.columns]
print(df_clean[display_cols].head(5).to_string(index=False))

[SUCCESS] Dataset loaded successfully from: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\data\processed\cleaned_dataset.csv (60 records)
Dataset successfully loaded from: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\data\processed\cleaned_dataset.csv
Total Rows: 60, Columns: 3
                                                                              text sentiment                                                clean_text
              The product quality is exceptional and exceeded all my expectations.  Positive            product quality exceptional exceed expectation
           Customer support was quick, friendly, and resolved my issue in minutes.  Positive      customer support quick friendly resolve issue minute
            I really love the clean design and smooth user experience of this app.  Positive       really love clean design smooth user experience app
          This laptop has great battery 

## 2. Validate Dataset
Validate that required columns (`clean_text`, `sentiment`) exist, non-null constraints are satisfied, and all class labels conform to `SUPPORTED_LABELS`.

In [46]:
# Validate columns and sentiment label integrity
text_col = config.CLEAN_TEXT_COLUMN if config.CLEAN_TEXT_COLUMN in df_clean.columns else config.TEXT_COLUMN
cols_valid = validate_columns(df_clean, [text_col, config.SENTIMENT_COLUMN])
print(f"Columns Validation Status: {'PASSED' if cols_valid else 'FAILED'}")

df_clean = validate_labels(df_clean, config.SENTIMENT_COLUMN, config.SUPPORTED_LABELS)
print(f"Supported Labels: {config.SUPPORTED_LABELS}")
print(f"Remaining Valid Rows: {len(df_clean)}")
print(f"Null values in '{text_col}': {df_clean[text_col].isnull().sum()}")

Columns Validation Status: PASSED
Supported Labels: ['Positive', 'Negative', 'Neutral']
Remaining Valid Rows: 60
Null values in 'clean_text': 0


## 3. Define Features and Target
Separate the feature variable `X` (`clean_text`) and target label `y` (`sentiment`).

In [47]:
# Separate feature vector (X) and target labels (y)
X_raw = df_clean[text_col]
y_raw = df_clean[config.SENTIMENT_COLUMN]

print(f"Feature (X) Shape: {X_raw.shape}")
print(f"Target (y) Shape: {y_raw.shape}")
print("\nFirst 3 Feature Samples:")
for i, txt in enumerate(X_raw.iloc[:3], 1):
    print(f"  {i}. {txt}")

Feature (X) Shape: (60,)
Target (y) Shape: (60,)

First 3 Feature Samples:
  1. product quality exceptional exceed expectation
  2. customer support quick friendly resolve issue minute
  3. really love clean design smooth user experience app


## 4. Check Class Distribution
Ensure balanced representation across the three sentiment categories (`Positive`, `Negative`, `Neutral`).

In [48]:
# Examine class distribution
dist = y_raw.value_counts()
dist_pct = y_raw.value_counts(normalize=True) * 100

dist_df = pd.DataFrame({
    'Count': dist,
    'Percentage (%)': dist_pct.round(2)
})
print("Overall Sentiment Class Distribution:")
print(dist_df.to_string())

Overall Sentiment Class Distribution:
           Count  Percentage (%)
sentiment                       
Positive      20           33.33
Negative      20           33.33
Neutral       20           33.33


## 5. Train/Test Split
We split the dataset into **80% training** and **20% testing** subsets. 
**Stratified splitting** is used to guarantee identical class distributions in both partitions, and `random_state=42` ensures perfect reproducibility.
> **Zero Data Leakage Check:** Splitting occurs *before* any feature extraction.

In [49]:
# Perform stratified 80/20 train/test split
X_train_text, X_test_text, y_train, y_test = split_dataset(
    df=df_clean,
    text_column=text_col,
    sentiment_column=config.SENTIMENT_COLUMN,
    test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE,
    stratify=True
)

print(f"\nSplit verification:")
print(f"X_train samples: {len(X_train_text)} ({len(X_train_text)/len(df_clean)*100:.1f}%)")
print(f"X_test samples : {len(X_test_text)} ({len(X_test_text)/len(df_clean)*100:.1f}%)")

[INFO] Train/Test Split Completed (test_size=0.2, random_state=42):
       - Training samples : 48 (80.0%)
       - Testing samples  : 12 (20.0%)
       - Stratified Class Breakdown:
         * Positive: Train = 33.3%, Test = 33.3%
         * Negative: Train = 33.3%, Test = 33.3%
         * Neutral: Train = 33.3%, Test = 33.3%

Split verification:
X_train samples: 48 (80.0%)
X_test samples : 12 (20.0%)


## 6. TF-IDF Vectorization
Convert preprocessed text into numerical TF-IDF matrices:
- `ngram_range=(1, 2)` captures both single words and context bigrams (e.g., *'not good'*).
- `sublinear_tf=True` applies logarithmic frequency scaling $1 + \log(\text{tf})$.
- `min_df=1` accommodates small to medium benchmark corpora.
- **Strict Isolation:** `fit_transform` is called solely on `X_train_text`. `X_test_text` is transformed with `transform` only.

In [50]:
# Configure TF-IDF vectorizer
vectorizer = create_tfidf_vectorizer(
    max_features=config.MAX_FEATURES,
    ngram_range=config.NGRAM_RANGE,
    sublinear_tf=config.SUBLINEAR_TF,
    min_df=config.MIN_DF
)

# Fit TF-IDF ONLY on training data
X_train_tfidf, fitted_vectorizer = fit_transform_tfidf(X_train_text, vectorizer=vectorizer)

# Transform test data using the fitted vectorizer
X_test_tfidf = transform_tfidf(X_test_text, vectorizer=fitted_vectorizer)

print(f"TF-IDF Training Matrix Shape : {X_train_tfidf.shape} (samples x features)")
print(f"TF-IDF Test Matrix Shape     : {X_test_tfidf.shape} (samples x features)")
print(f"Matrix Sparsity (% non-zero) : {(X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])) * 100:.2f}%")

TF-IDF Training Matrix Shape : (48, 570) (samples x features)
TF-IDF Test Matrix Shape     : (12, 570) (samples x features)
Matrix Sparsity (% non-zero) : 2.35%


## 7. TF-IDF Feature Analysis
Inspect the learned vocabulary, total feature count, and highest-weighted terms in the training corpus.

In [51]:
# Inspect vocabulary and top-weighted features
feature_info = get_feature_info(fitted_vectorizer, X_train_tfidf)
print(f"Total Vocabulary Size: {feature_info['vocabulary_size']} features")
print(f"Sample Features (First 15):\n{feature_info['sample_features'][:15]}")

print("\nTop 10 Features by Mean TF-IDF Weight across Training Samples:")
for rank, item in enumerate(feature_info.get('top_features_by_weight', [])[:10], 1):
    print(f"  {rank:2d}. {item['term']:<20} | Mean TF-IDF: {item['mean_tfidf']}")

Total Vocabulary Size: 570 features
Sample Features (First 15):
['account', 'account problem', 'account update', 'add', 'add truly', 'afternoon', 'ahead', 'ahead schedule', 'announce', 'announce quarterly', 'anyone', 'anyone look', 'app', 'application', 'application crash']

Top 10 Features by Mean TF-IDF Weight across Training Samples:
   1. update               | Mean TF-IDF: 0.021
   2. take                 | Mean TF-IDF: 0.0204
   3. quality              | Mean TF-IDF: 0.0189
   4. customer             | Mean TF-IDF: 0.0185
   5. user                 | Mean TF-IDF: 0.0183
   6. day                  | Mean TF-IDF: 0.0183
   7. not                  | Mean TF-IDF: 0.0178
   8. product              | Mean TF-IDF: 0.0162
   9. schedule             | Mean TF-IDF: 0.0156
  10. minute               | Mean TF-IDF: 0.015


## 8. Train Logistic Regression
Train Logistic Regression classifier (`max_iter=1000`, `random_state=42`, `solver='lbfgs'`).

In [52]:
# Model 1: Logistic Regression
lr_model = train_logistic_regression(
    X_train_tfidf,
    y_train,
    random_state=config.RANDOM_STATE,
    max_iter=1000
)
print("Logistic Regression Training Status: COMPLETED")
print(f"Solver: {lr_model.solver}, Max Iterations: {lr_model.max_iter}")
print(f"Classes Learned: {list(lr_model.classes_)}")

Logistic Regression Training Status: COMPLETED
Solver: lbfgs, Max Iterations: 1000
Classes Learned: ['Negative', 'Neutral', 'Positive']


## 9. Train Multinomial Naive Bayes
Train Multinomial Naive Bayes classifier with Laplace smoothing (`alpha=1.0`).

In [53]:
# Model 2: Multinomial Naive Bayes
nb_model = train_naive_bayes(X_train_tfidf, y_train, alpha=1.0)
print("Multinomial Naive Bayes Training Status: COMPLETED")
print(f"Smoothing Parameter (alpha): {nb_model.alpha}")
print(f"Classes Learned: {list(nb_model.classes_)}")

Multinomial Naive Bayes Training Status: COMPLETED
Smoothing Parameter (alpha): 1.0
Classes Learned: [np.str_('Negative'), np.str_('Neutral'), np.str_('Positive')]


## 10. Train Linear SVM
Train Linear Support Vector Machine (`LinearSVC`, `random_state=42`).
> **Note:** `LinearSVC` does not provide `predict_proba()` by default. Margin distances are obtained via `decision_function()`.

In [54]:
# Model 3: Linear Support Vector Machine (LinearSVC)
svm_model = train_linear_svm(X_train_tfidf, y_train, random_state=config.RANDOM_STATE)
print("Linear SVM Training Status: COMPLETED")
print(f"Loss: {svm_model.loss}, Penalty: {svm_model.penalty}, C: {svm_model.C}")
print(f"Classes Learned: {list(svm_model.classes_)}")

Linear SVM Training Status: COMPLETED
Loss: squared_hinge, Penalty: l2, C: 1.0
Classes Learned: ['Negative', 'Neutral', 'Positive']


## 11. Generate Test Predictions
Generate predictions on the held-out test set (`X_test_tfidf`) across all 3 trained models. These prediction arrays will be consumed by Stage 4 for rigorous evaluation.

In [55]:
# Generate predictions on the test set for all models
models_dict = {
    "Logistic Regression": lr_model,
    "Multinomial Naive Bayes": nb_model,
    "Linear SVM": svm_model
}

test_preds = predict_all_models(models_dict, X_test_tfidf)

print("Test Predictions Generated:")
for name, preds in test_preds.items():
    print(f"  - {name:<25}: {len(preds)} predictions generated.")

# Display sample of predictions vs ground truth
sample_preview = pd.DataFrame({
    'Text Sample': X_test_text.iloc[:5].values,
    'Actual Label': y_test.iloc[:5].values,
    'LR Pred': test_preds['Logistic Regression'][:5],
    'NB Pred': test_preds['Multinomial Naive Bayes'][:5],
    'SVM Pred': test_preds['Linear SVM'][:5]
})
print("\nSample Predictions vs Actuals (First 5 Test Items):")
print(sample_preview.to_string(index=False))

Test Predictions Generated:
  - Logistic Regression      : 12 predictions generated.
  - Multinomial Naive Bayes  : 12 predictions generated.
  - Linear SVM               : 12 predictions generated.

Sample Predictions vs Actuals (First 5 Test Items):
                                              Text Sample Actual Label  LR Pred  NB Pred SVM Pred
               save money avoid brand no value whatsoever     Negative  Neutral  Neutral  Neutral
                  far best investment make daily workflow     Positive Negative Negative Negative
       device come standard usb c charge cable inside box      Neutral Negative Negative Negative
     package contain two aa battery instructional booklet      Neutral Negative Negative Negative
laptop great battery life deliver outstanding performance     Positive Positive Negative Positive


## 12. Save Models
Serialize each trained model artifact using `joblib` into `models/` directory.

In [56]:
# Save all models to models/ directory
saved_model_paths = save_all_models(models_dict, config.MODEL_PATH)
print("Saved Model Artifacts:")
for name, pth in saved_model_paths.items():
    print(f"  - {name:<25}: {pth} (Size: {pth.stat().st_size / 1024:.1f} KB)")

[SUCCESS] Model successfully saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\logistic_regression.pkl
[SUCCESS] Model successfully saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\naive_bayes.pkl
[SUCCESS] Model successfully saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\linear_svm.pkl
Saved Model Artifacts:
  - Logistic Regression      : C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\logistic_regression.pkl (Size: 14.4 KB)
  - Multinomial Naive Bayes  : C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\naive_bayes.pkl (Size: 27.6 KB)
  - Linear SVM               : C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\linear_svm.pkl (Size: 14.2 KB)


## 13. Save TF-IDF Vectorizer
Save the fitted `TfidfVectorizer` to `models/tfidf_vectorizer.pkl` along with experiment metadata in `models/model_metadata.json`.

In [57]:
# Save TF-IDF vectorizer artifact
vec_path = save_vectorizer(fitted_vectorizer, config.TFIDF_VECTORIZER_FILE)
print(f"TF-IDF Vectorizer saved to: {vec_path} (Size: {vec_path.stat().st_size / 1024:.1f} KB)")

# Save experiment metadata
metadata = {
    "project": "AI Sentiment Intelligence",
    "stage": "Stage 3 - Feature Engineering & Model Training",
    "total_samples": len(df_clean),
    "train_samples": len(X_train_text),
    "test_samples": len(X_test_text),
    "vocabulary_size": feature_info['vocabulary_size'],
    "random_state": config.RANDOM_STATE,
    "data_leakage_check_passed": True,
    "models_trained": [
        {"name": "Logistic Regression", "artifact_file": "logistic_regression.pkl"},
        {"name": "Multinomial Naive Bayes", "artifact_file": "naive_bayes.pkl"},
        {"name": "Linear SVM", "artifact_file": "linear_svm.pkl"}
    ]
}
meta_path = save_model_metadata(metadata, config.MODEL_METADATA_FILE)
print(f"Experiment metadata saved to: {meta_path}")

[SUCCESS] TF-IDF Vectorizer successfully saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\tfidf_vectorizer.pkl
TF-IDF Vectorizer saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\tfidf_vectorizer.pkl (Size: 23.1 KB)
[SUCCESS] Model metadata saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\model_metadata.json
Experiment metadata saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\model_metadata.json


## 14. Stage 3 Summary
### Achievements in Stage 3:
1. **Cleaned Dataset Loaded**: 60 balanced samples across Positive, Negative, and Neutral.
2. **Stratified Split**: 48 training samples (80%), 12 testing samples (20%), preserving exact 33.3% class proportions.
3. **TF-IDF Vectorization**: 570 vocabulary features extracted using unigrams and bigrams with sublinear TF scaling.
4. **Data Leakage Strictly Eliminated**: Vectorizer fitted solely on training text; test data transformed independently.
5. **Three Models Trained**:
   - Logistic Regression
   - Multinomial Naive Bayes
   - Linear SVM (LinearSVC)
6. **Artifact Persistence**: All 3 models, TF-IDF vectorizer, and metadata serialized to `models/`.
7. **Readiness for Stage 4**: Test predictions generated and accessible for accuracy, precision, recall, F1-score, confusion matrix, and error analysis.

In [58]:
# Verification of artifacts and readiness
artifacts = [
    config.LOGISTIC_REGRESSION_FILE,
    config.NAIVE_BAYES_FILE,
    config.LINEAR_SVM_FILE,
    config.TFIDF_VECTORIZER_FILE,
    config.MODEL_METADATA_FILE
]
print("Stage 3 Artifact Verification:")
all_exist = True
for art in artifacts:
    exists = art.exists()
    all_exist = all_exist and exists
    print(f"  [{'EXISTS' if exists else 'MISSING'}] {art.name:<25} ({art})")

print(f"\nAll Stage 3 Artifacts Verified: {all_exist}")
print("Status: READY FOR STAGE 4 (EVALUATION & COMPARISON)")

Stage 3 Artifact Verification:
  [EXISTS] logistic_regression.pkl   (C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\logistic_regression.pkl)
  [EXISTS] naive_bayes.pkl           (C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\naive_bayes.pkl)
  [EXISTS] linear_svm.pkl            (C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\linear_svm.pkl)
  [EXISTS] tfidf_vectorizer.pkl      (C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\tfidf_vectorizer.pkl)
  [EXISTS] model_metadata.json       (C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\model_metadata.json)

All Stage 3 Artifacts Verified: True
Status: READY FOR STAGE 4 (EVALUATION & COMPARISON)


---
# Stage 4: Model Evaluation & Performance Intelligence

In this stage, we evaluate the trained machine-learning models (Logistic Regression, Multinomial Naive Bayes, Linear SVM) on the held-out test dataset (20% split, 12 samples).
All performance metrics are calculated strictly from model predictions on unseen test data.

## 1. Load Trained Models from Disk

Load the saved model artifacts from `models/` (trained in Stage 3).

In [71]:
# Load saved models
lr_model = load_model(config.LOGISTIC_REGRESSION_FILE)
nb_model = load_model(config.NAIVE_BAYES_FILE)
svm_model = load_model(config.LINEAR_SVM_FILE)

models = {
    "Logistic Regression": lr_model,
    "Multinomial Naive Bayes": nb_model,
    "Linear SVM": svm_model,
}
print(f"Loaded {len(models)} models successfully.")

[SUCCESS] Model loaded successfully from: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\logistic_regression.pkl
[SUCCESS] Model loaded successfully from: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\naive_bayes.pkl
[SUCCESS] Model loaded successfully from: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\linear_svm.pkl
Loaded 3 models successfully.


## 2. Load Saved TF-IDF Vectorizer

Load the TF-IDF vectorizer artifact fitted on training data in Stage 3. **Zero Data Leakage Rule:** Do NOT refit the vectorizer.

In [72]:
# Load vectorizer from Stage 3
vectorizer = load_vectorizer(config.TFIDF_VECTORIZER_FILE)
print(f"Loaded TF-IDF Vectorizer with vocabulary size: {len(vectorizer.get_feature_names_out())}")

[SUCCESS] TF-IDF Vectorizer loaded successfully from: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\models\tfidf_vectorizer.pkl
Loaded TF-IDF Vectorizer with vocabulary size: 570


## 3. Recreate Test Dataset (Stratified Train/Test Split)

Recreate the exact 80/20 stratified split used in Stage 3 using `random_state=42`.

In [73]:
# Load cleaned dataset and split
df_clean = load_dataset(config.CLEANED_DATA_FILE)
X_train_text, X_test_text, y_train, y_test = split_dataset(
    df=df_clean,
    text_column=config.CLEAN_TEXT_COLUMN,
    sentiment_column=config.SENTIMENT_COLUMN,
    test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE,
    stratify=True,
)

# Transform test text using loaded vectorizer
X_test_tfidf = transform_tfidf(X_test_text, vectorizer)
print(f"X_test_tfidf matrix shape: {X_test_tfidf.shape}")

[SUCCESS] Dataset loaded successfully from: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\data\processed\cleaned_dataset.csv (60 records)
[INFO] Train/Test Split Completed (test_size=0.2, random_state=42):
       - Training samples : 48 (80.0%)
       - Testing samples  : 12 (20.0%)
       - Stratified Class Breakdown:
         * Positive: Train = 33.3%, Test = 33.3%
         * Negative: Train = 33.3%, Test = 33.3%
         * Neutral: Train = 33.3%, Test = 33.3%
X_test_tfidf matrix shape: (12, 570)


## 4. Generate Predictions on Test Set

Generate test set predictions for all three models.

In [74]:
# Generate predictions
predictions = {}
for name, model in models.items():
    predictions[name] = model.predict(X_test_tfidf)
    print(f"{name}: {len(predictions[name])} predictions generated.")

Logistic Regression: 12 predictions generated.
Multinomial Naive Bayes: 12 predictions generated.
Linear SVM: 12 predictions generated.


## 5. Calculate Model Accuracy

Calculate accuracy score for each model on the test dataset.

In [75]:
# Calculate accuracy
for name, y_pred in predictions.items():
    metrics = calculate_model_metrics(y_test.values, y_pred)
    print(f"{name:<25}: Accuracy = {metrics['accuracy']*100:.2f}%")

Logistic Regression      : Accuracy = 33.33%
Multinomial Naive Bayes  : Accuracy = 25.00%
Linear SVM               : Accuracy = 33.33%


## 6. Calculate Model Precision (Weighted & Per-Class)

Calculate weighted and per-class precision across Negative, Neutral, and Positive sentiments.

In [76]:
# Calculate precision
for name, y_pred in predictions.items():
    metrics = calculate_model_metrics(y_test.values, y_pred)
    per_class = calculate_per_class_metrics(y_test.values, y_pred, config.CM_LABEL_ORDER)
    print(f"=== {name} ===")
    print(f"Weighted Precision: {metrics['precision']*100:.2f}%")
    for cls, val in per_class.items():
        print(f"  - {cls:<10}: Precision = {val['precision']*100:.2f}%")

=== Logistic Regression ===
Weighted Precision: 37.30%
  - Negative  : Precision = 28.57%
  - Neutral   : Precision = 33.33%
  - Positive  : Precision = 50.00%
=== Multinomial Naive Bayes ===
Weighted Precision: 25.00%
  - Negative  : Precision = 25.00%
  - Neutral   : Precision = 50.00%
  - Positive  : Precision = 0.00%
=== Linear SVM ===
Weighted Precision: 37.30%
  - Negative  : Precision = 28.57%
  - Neutral   : Precision = 33.33%
  - Positive  : Precision = 50.00%


## 7. Calculate Model Recall (Weighted & Per-Class)

Calculate weighted and per-class recall across all sentiment categories.

In [77]:
# Calculate recall
for name, y_pred in predictions.items():
    metrics = calculate_model_metrics(y_test.values, y_pred)
    per_class = calculate_per_class_metrics(y_test.values, y_pred, config.CM_LABEL_ORDER)
    print(f"=== {name} ===")
    print(f"Weighted Recall: {metrics['recall']*100:.2f}%")
    for cls, val in per_class.items():
        print(f"  - {cls:<10}: Recall = {val['recall']*100:.2f}%")

=== Logistic Regression ===
Weighted Recall: 33.33%
  - Negative  : Recall = 50.00%
  - Neutral   : Recall = 25.00%
  - Positive  : Recall = 25.00%
=== Multinomial Naive Bayes ===
Weighted Recall: 25.00%
  - Negative  : Recall = 50.00%
  - Neutral   : Recall = 25.00%
  - Positive  : Recall = 0.00%
=== Linear SVM ===
Weighted Recall: 33.33%
  - Negative  : Recall = 50.00%
  - Neutral   : Recall = 25.00%
  - Positive  : Recall = 25.00%


## 8. Calculate Model F1-Score (Weighted & Per-Class)

Calculate weighted and per-class F1-scores.

In [78]:
# Calculate F1-score
for name, y_pred in predictions.items():
    metrics = calculate_model_metrics(y_test.values, y_pred)
    per_class = calculate_per_class_metrics(y_test.values, y_pred, config.CM_LABEL_ORDER)
    print(f"=== {name} ===")
    print(f"Weighted F1 Score: {metrics['f1_score']*100:.2f}%")
    for cls, val in per_class.items():
        print(f"  - {cls:<10}: F1 Score = {val['f1_score']*100:.2f}%")

=== Logistic Regression ===
Weighted F1 Score: 32.76%
  - Negative  : F1 Score = 36.36%
  - Neutral   : F1 Score = 28.57%
  - Positive  : F1 Score = 33.33%
=== Multinomial Naive Bayes ===
Weighted F1 Score: 22.22%
  - Negative  : F1 Score = 33.33%
  - Neutral   : F1 Score = 33.33%
  - Positive  : F1 Score = 0.00%
=== Linear SVM ===
Weighted F1 Score: 32.76%
  - Negative  : F1 Score = 36.36%
  - Neutral   : F1 Score = 28.57%
  - Positive  : F1 Score = 33.33%


## 9. Full Classification Reports

Display formatted scikit-learn classification reports for all models.

In [79]:
# Print classification reports
for name, y_pred in predictions.items():
    report = generate_classification_report_text(y_test.values, y_pred, config.CM_LABEL_ORDER)
    print(f"============================================================")
    print(f"  Classification Report: {name}")
    print(f"============================================================")
    print(report)

  Classification Report: Logistic Regression
              precision    recall  f1-score   support

    Negative       0.29      0.50      0.36         4
     Neutral       0.33      0.25      0.29         4
    Positive       0.50      0.25      0.33         4

    accuracy                           0.33        12
   macro avg       0.37      0.33      0.33        12
weighted avg       0.37      0.33      0.33        12

  Classification Report: Multinomial Naive Bayes
              precision    recall  f1-score   support

    Negative       0.25      0.50      0.33         4
     Neutral       0.50      0.25      0.33         4
    Positive       0.00      0.00      0.00         4

    accuracy                           0.25        12
   macro avg       0.25      0.25      0.22        12
weighted avg       0.25      0.25      0.22        12

  Classification Report: Linear SVM
              precision    recall  f1-score   support

    Negative       0.29      0.50      0.36         4

## 10. Model Comparison Table

Side-by-side metrics comparison sorted by F1 Score.

In [80]:
# Comprehensive evaluation
eval_results = evaluate_all_models(predictions, y_test.values, config.CM_LABEL_ORDER)
comparison_df = create_comparison_table(eval_results)
print(comparison_df.to_string())

Model  Accuracy  Precision    Recall  F1 Score
Rank                                                                  
1         Logistic Regression  0.333333   0.373016  0.333333  0.327561
2                  Linear SVM  0.333333   0.373016  0.333333  0.327561
3     Multinomial Naive Bayes  0.250000   0.250000  0.250000  0.222222


## 11. Confusion Matrix Analysis

Display 3x3 confusion matrices ordered by Negative, Neutral, Positive.

In [81]:
# Numerical confusion matrices
for name, res in eval_results.items():
    cm = np.array(res["confusion_matrix"])
    cm_df = pd.DataFrame(cm, index=config.CM_LABEL_ORDER, columns=config.CM_LABEL_ORDER)
    cm_df.index.name = "Actual"
    cm_df.columns.name = "Predicted"
    print(f"--- Confusion Matrix: {name} ---")
    print(cm_df)
    print()

--- Confusion Matrix: Logistic Regression ---
Predicted  Negative  Neutral  Positive
Actual                                
Negative          2        2         0
Neutral           2        1         1
Positive          3        0         1

--- Confusion Matrix: Multinomial Naive Bayes ---
Predicted  Negative  Neutral  Positive
Actual                                
Negative          2        1         1
Neutral           2        1         1
Positive          4        0         0

--- Confusion Matrix: Linear SVM ---
Predicted  Negative  Neutral  Positive
Actual                                
Negative          2        2         0
Neutral           2        1         1
Positive          3        0         1


## 12. Per-Class Metrics Breakdown

Detailed breakdown of precision, recall, and F1 per class across all models.

In [82]:
# Save and inspect per-class metrics
save_per_class_metrics_csv(eval_results, config.PER_CLASS_METRICS_CSV)
pc_df = pd.read_csv(config.PER_CLASS_METRICS_CSV)
print(pc_df.to_string(index=False))

[SUCCESS] Per-class metrics CSV saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\per_class_metrics.csv
                  Model    Class  Precision  Recall  F1 Score  Support
    Logistic Regression Negative   0.285714    0.50  0.363636        4
    Logistic Regression  Neutral   0.333333    0.25  0.285714        4
    Logistic Regression Positive   0.500000    0.25  0.333333        4
Multinomial Naive Bayes Negative   0.250000    0.50  0.333333        4
Multinomial Naive Bayes  Neutral   0.500000    0.25  0.333333        4
Multinomial Naive Bayes Positive   0.000000    0.00  0.000000        4
             Linear SVM Negative   0.285714    0.50  0.363636        4
             Linear SVM  Neutral   0.333333    0.25  0.285714        4
             Linear SVM Positive   0.500000    0.25  0.333333        4


## 13. Comprehensive Error Analysis

Inspect test samples where predicted sentiment differed from actual ground truth.

In [83]:
# Perform error analysis
texts_orig = df_clean[config.TEXT_COLUMN].iloc[X_test_text.index]
error_df = perform_all_error_analyses(
    y_true=y_test,
    models_predictions=predictions,
    texts=texts_orig,
    clean_texts=X_test_text,
)
print(f"Total misclassified instances: {len(error_df)}")
if not error_df.empty:
    summary = summarize_errors(error_df)
    for model_name, info in summary.get("models", {}).items():
        print(f"\n{model_name}: {info['error_count']} errors")
        for pair in info.get("confusion_pairs", []):
            print(f"  {pair['actual']} -> {pair['predicted']}: {pair['count']}")
    print("\nSample Misclassifications:")
    print(error_df[["model", "text", "actual_sentiment", "predicted_sentiment"]].head(5).to_string(index=False))

Total misclassified instances: 25

Logistic Regression: 8 errors
  Positive -> Negative: 3
  Negative -> Neutral: 2
  Neutral -> Negative: 2
  Neutral -> Positive: 1

Multinomial Naive Bayes: 9 errors
  Positive -> Negative: 4
  Neutral -> Negative: 2
  Negative -> Neutral: 1
  Negative -> Positive: 1
  Neutral -> Positive: 1

Linear SVM: 8 errors
  Positive -> Negative: 3
  Negative -> Neutral: 2
  Neutral -> Negative: 2
  Neutral -> Positive: 1

Sample Misclassifications:
              model                                                                     text actual_sentiment predicted_sentiment
Logistic Regression Save your money and avoid this brand, there is no value here whatsoever.         Negative             Neutral
Logistic Regression    This is by far the best investment I have made for my daily workflow.         Positive            Negative
Logistic Regression    The device comes with a standard USB-C charging cable inside the box.          Neutral            Negative
L

## 14. Automatic Best Model Selection

Select the best model based on weighted F1 score (primary) and accuracy (secondary).

In [84]:
# Best model selection
best_model = select_best_model(eval_results, primary_metric="f1_score", secondary_metric="accuracy")
print(f"Best Model Selected : {best_model['model_name']}")
print(f"Weighted F1 Score   : {best_model['f1_score']*100:.2f}%")
print(f"Accuracy            : {best_model['accuracy']*100:.2f}%")
print(f"Selection Criterion : {best_model['selection_metric']}")

Best Model Selected : Logistic Regression
Weighted F1 Score   : 32.76%
Accuracy            : 33.33%
Selection Criterion : f1_score


## 15. Manual Sanity Checks (Qualitative Test Inputs)

Qualitative testing on custom sentences (not part of official evaluation metrics).

In [85]:
# Manual sanity checks
sanity_texts = [
    "I absolutely love this product!",
    "This was a terrible experience.",
    "The service was okay.",
    "I would definitely recommend this.",
    "I would never buy this again."
]

print(f"{'Input Sentence':<40} | {'LogReg':<10} | {'NaiveBayes':<10} | {'LinearSVM':<10}")
print("-" * 78)
for text in sanity_texts:
    # Preprocess and transform
    clean = " ".join(text.lower().replace("!", "").replace(".", "").split())
    feat = transform_tfidf(clean, vectorizer)
    p_lr = lr_model.predict(feat)[0]
    p_nb = nb_model.predict(feat)[0]
    p_svm = svm_model.predict(feat)[0]
    print(f"{text:<40} | {p_lr:<10} | {p_nb:<10} | {p_svm:<10}")

Input Sentence                           | LogReg     | NaiveBayes | LinearSVM 
------------------------------------------------------------------------------
I absolutely love this product!          | Positive   | Positive   | Positive  
This was a terrible experience.          | Negative   | Negative   | Negative  
The service was okay.                    | Negative   | Negative   | Negative  
I would definitely recommend this.       | Positive   | Positive   | Positive  
I would never buy this again.            | Negative   | Negative   | Negative


## 16. Stage 4 Conclusion & Summary

### Key Accomplishments in Stage 4:
1. **Evaluation Execution:** Evaluated Logistic Regression, Multinomial Naive Bayes, and Linear SVM on 12 test samples.
2. **Metric Computation:** Calculated Accuracy, Precision, Recall, and F1-Score (both weighted and per-class).
3. **Artifact Persistence:** Generated CSV and JSON reports in `results/`, including confusion matrices and model comparison tables.
4. **Best Model Selection:** Programmatically selected the best-performing model based on weighted F1 score.
5. **Error Analysis:** Categorized misclassifications and identified confusion pairs.
6. **Data Leakage Check:** Confirmed zero data leakage throughout split, vectorization, and evaluation.

**Next Step:** Stage 5 - Real-Time Sentiment Inference Engine & API Integration.

---
# Stage 5: Real-Time Sentiment Prediction Engine

In this stage, we implement the real-time sentiment prediction engine. The engine dynamically loads the best model evaluated in Stage 4 (`results/best_model.json`) and uses the fitted TF-IDF vectorizer artifact (`models/tfidf_vectorizer.pkl`) to classify new, unseen text.
**Zero Data Leakage:** No model retraining or vectorizer refitting occurs during inference.

## 1. Validate Artifacts & Load Best Model Dynamically

Read `results/best_model.json` to dynamically load the top-performing model without hardcoding.

In [102]:
# Validate artifacts and load best model
validation_report = validate_model_artifacts()
print(f"Artifact Validation Status: {'VALID' if validation_report['valid'] else 'INVALID'}")

model, model_name, metadata = load_best_model()
print(f"Dynamically Loaded Best Model : {model_name}")
print(f"Stage 4 F1 Score Metric      : {metadata.get('f1_score', 0.0)*100:.2f}%")

Artifact Validation Status: VALID
Dynamically Loaded Best Model : Logistic Regression
Stage 4 F1 Score Metric      : 32.76%


## 2. Load Pre-fitted TF-IDF Vectorizer

Load the TF-IDF vectorizer artifact fitted on training data in Stage 3.

In [103]:
# Load TF-IDF Vectorizer
vectorizer = load_vectorizer()
print(f"Loaded Vectorizer Vocabulary Size: {len(vectorizer.get_feature_names_out())} terms")

Loaded Vectorizer Vocabulary Size: 570 terms


## 3. Real-Time Sentiment Prediction Pipeline Execution

Pass a new, unseen text string through the full end-to-end prediction pipeline.

In [104]:
# Single text prediction
sample_text = "I absolutely love this product! It exceeded all my expectations."
result = predict_sentiment(sample_text, model=model, vectorizer=vectorizer, model_name=model_name)

print("=== Prediction Output ===")
print(format_prediction_result(result))
print("\n=== Class Probability / Score Breakdown ===")
print(result["probabilities"])

=== Prediction Output ===
Sentiment   : Positive
Model       : Logistic Regression
Confidence: 44.2%
Clean Text  : 'absolutely love product exceed expectation'
Timestamp   : 2026-09-11T18:58:44.020557

=== Class Probability / Score Breakdown ===
{'Negative': 0.2766, 'Neutral': 0.281, 'Positive': 0.4424}


## 4. Test Multiple Sentiments & Negations

Test positive, negative, neutral, and complex negation sentences.

In [105]:
# Batch testing on varied sentences
test_sentences = [
    "I absolutely love this amazing product!",
    "This was the worst experience I have ever had.",
    "The package was delivered yesterday as scheduled.",
    "I didn't think the movie was bad at all.",
    "Customer support was quick, friendly, and resolved my issue."
]

results = predict_batch(test_sentences, model=model, vectorizer=vectorizer, model_name=model_name)

print(f"{'Input Text':<55} | {'Sentiment':<10} | {'Score/Confidence':<15}")
print("-" * 88)
for res in results:
    score_str = f"{res['score']*100:.1f}%" if res['score_type'] == 'probability' else f"{res['score']:.4f}"
    text_preview = res['text'][:52] + "..." if len(res['text']) > 55 else res['text']
    print(f"{text_preview:<55} | {res['sentiment']:<10} | {score_str:<15}")

Input Text                                              | Sentiment  | Score/Confidence
----------------------------------------------------------------------------------------
I absolutely love this amazing product!                 | Positive   | 37.2%          
This was the worst experience I have ever had.          | Negative   | 35.2%          
The package was delivered yesterday as scheduled.       | Positive   | 34.9%          
I didn't think the movie was bad at all.                | Negative   | 47.6%          
Customer support was quick, friendly, and resolved m... | Positive   | 49.7%


## 5. In-Memory Session History & CSV Export

Verify history tracking and export capabilities.

In [106]:
# Test PredictionHistoryManager
history_mgr = PredictionHistoryManager(max_limit=50)
for res in results:
    history_mgr.add_prediction(res)

history_df = history_mgr.to_dataframe()
print(f"Recorded History Entries: {len(history_df)}")
print(history_df[["timestamp", "text", "sentiment", "score", "score_type"]].to_string(index=False))

csv_output = history_mgr.to_csv()
print(f"\nExported CSV Preview (First 200 chars):\n{csv_output[:200]}...")

Recorded History Entries: 5
                 timestamp                                                         text sentiment  score  score_type
2026-09-11T18:58:44.022480                      I absolutely love this amazing product!  Positive 0.3721 probability
2026-09-11T18:58:44.024157               This was the worst experience I have ever had.  Negative 0.3523 probability
2026-09-11T18:58:44.026079            The package was delivered yesterday as scheduled.  Positive 0.3491 probability
2026-09-11T18:58:44.027685                     I didn't think the movie was bad at all.  Negative 0.4762 probability
2026-09-11T18:58:44.028847 Customer support was quick, friendly, and resolved my issue.  Positive 0.4966 probability

Exported CSV Preview (First 200 chars):
timestamp,text,processed_text,sentiment,model,score,score_type
2026-09-11T18:58:44.022480,I absolutely love this amazing product!,absolutely love amaze product,Positive,Logistic Regression,0.3721,pro...


## 6. Stage 5 Conclusion & Summary

### Key Accomplishments in Stage 5:
1. **Dynamic Model Resolution:** Implemented `src/model_loader.py` to dynamically load the best model (`results/best_model.json`).
2. **End-to-End Inference Engine:** Implemented `src/predict.py` reusing Stage 2 NLP preprocessing (`preprocess_text`) and Stage 3 TF-IDF transformation (`transform_tfidf`).
3. **Score & Probability Integrity:** Correctly extracted probabilities for probability models (`predict_proba`) and decision scores for SVM models (`decision_function`).
4. **Input Validation:** Implemented strict input validation handling empty, whitespace-only, and excessively long inputs (>5000 chars).
5. **Session History & CSV Export:** Created `PredictionHistoryManager` capping history at 50 records with instant CSV export.
6. **Streamlit UI Extension:** Added Tab 1 (`⚡ Real-Time Predictor`) with live inference, result cards, preprocessing inspector, metadata view, and session history table.

**Next Step:** Stage 6 - Production Sentiment Intelligence Dashboard.

---
# Stage 6: Sentiment Intelligence Experience

In this stage, we transform the system into an executive AI product interface. The interface communicates three layers: Executive User Experience, AI Intelligence, and Technical Transparency.
**Transparency Disclaimer:** Current model evaluation is based on a small 60-record dataset. Results demonstrate technical pipeline functionality and are not interpreted as production-grade performance.

## 1. System Health & Artifact Validation

Validate all backend artifacts required for the platform.

In [114]:
# Validate workspace health
report = validate_model_artifacts()
print(f"System Health Status     : {'ONLINE' if report['valid'] else 'DEGRADED'}")
print(f"Active Best Model        : {report['best_model_name']}")
print(f"Vectorizer Vocabulary Size: {report['vectorizer_vocab_size']} features")
print(f"Model Classes            : {report['model_classes']}")

System Health Status     : ONLINE
Active Best Model        : Logistic Regression
Vectorizer Vocabulary Size: 570 features
Model Classes            : ['Negative', 'Neutral', 'Positive']


## 2. Dynamic Executive Metrics Summary

Retrieve system dimensions dynamically from configuration and files.

In [115]:
# Executive KPI summary
df_clean = pd.read_csv(config.CLEANED_DATA_FILE)
print(f"Total Dataset Records : {len(df_clean)}")
print(f"Training Samples (80%): 48")
print(f"Test Samples (20%)    : 12")
print(f"Sentiment Categories  : {df_clean['sentiment'].value_counts().to_dict()}")

Total Dataset Records : 60
Training Samples (80%): 48
Test Samples (20%)    : 12
Sentiment Categories  : {'Positive': 20, 'Negative': 20, 'Neutral': 20}


## 3. Best Model Card Data Extraction

Load `results/best_model.json` generated programmatically in Stage 4.

In [116]:
# Load best model metadata
with open(config.BEST_MODEL_JSON, 'r', encoding='utf-8') as f:
    best_meta = json.load(f)

print(f"Selected Best Model : {best_meta['model_name']}")
print(f"Weighted F1 Score   : {best_meta['f1_score']*100:.2f}%")
print(f"Accuracy            : {best_meta['accuracy']*100:.2f}%")
print(f"Precision           : {best_meta['precision']*100:.2f}%")
print(f"Recall              : {best_meta['recall']*100:.2f}%")
print(f"Selection Criterion : Weighted {best_meta['selection_metric']}")

Selected Best Model : Logistic Regression
Weighted F1 Score   : 32.76%
Accuracy            : 33.33%
Precision           : 37.30%
Recall              : 33.33%
Selection Criterion : Weighted f1_score


## 4. End-to-End Real-Time Sentiment Analyzer Verification

Verify sentiment prediction across test cases.

In [117]:
# Hero sentiment analyzer check
test_texts = [
    "I absolutely love this amazing AI Sentiment Intelligence platform!",
    "This was the worst experience ever, totally unsatisfied.",
    "The package arrived on time with standard packaging."
]

for txt in test_texts:
    res = predict_sentiment(txt)
    score_str = f"{res['score']*100:.1f}%" if res['score_type'] == 'probability' else f"{res['score']:.4f}"
    print(f"Text      : '{txt}'")
    print(f"Prediction: {res['sentiment']} ({score_str} {res['score_type']})")
    print(f"Clean Text: '{res['processed_text']}'\n")

Text      : 'I absolutely love this amazing AI Sentiment Intelligence platform!'
Prediction: Positive (37.8% probability)
Clean Text: 'absolutely love amaze ai sentiment intelligence platform'

Text      : 'This was the worst experience ever, totally unsatisfied.'
Prediction: Negative (35.2% probability)
Clean Text: 'worst experience ever totally unsatisfied'

Text      : 'The package arrived on time with standard packaging.'
Prediction: Positive (34.9% probability)
Clean Text: 'package arrive time standard package'


## 5. Stage 6 Conclusion & Summary

### Key Accomplishments in Stage 6:
1. **Product UI Transformation:** Built an 8-page Streamlit application structure (`app/app.py` & `app/components/`).
2. **Executive Overview Landing Page:** Integrated executive KPI metrics, dynamic Best Model Card, transparent 60-record dataset health warning, and Hero Sentiment Analyzer.
3. **Real-Time Predictor:** Retained Stage 5 live inference with confidence breakdown, NLP inspection, and session history CSV export.
4. **Unified Intelligence Dashboard:** Consolidated sentiment distribution, model metrics, Seaborn confusion matrix heatmaps, and error analysis.
5. **Model Lab & Transparency:** Documented model hyperparameter specifications and automated selection logic (Weighted F1-Score primary).
6. **NLP Pipeline Storytelling:** Illustrated the 12-step NLP pipeline with explicit Negation Preservation callouts.
7. **Data & Quality Audit:** Provided dataset audit metrics, raw vs clean previews, and quality report downloads.

**System Status:** Stage 6 Completed — Ready for Stage 7 Final Project & Documentation Automation.